# Track forecast comparison

Load Ragasa and Yagi forecasts, align them with the JMA best tracks, validate the expected time points, and export one two-panel figure per storm.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

INPUT_DIR = Path("./input")
OUTPUT_DIR = Path("./output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_STYLES = {
    "era5": {"label": "ERA5_WRF", "color": "#4c78a8", "marker": "x"},
    "pangu": {"label": "Pangu_WRF", "color": "#f58518", "marker": "s"},
    "graphcast": {"label": "GraphCast_WRF", "color": "#54a24b", "marker": "^"},
    "fengwu": {"label": "FengWu_WRF", "color": "#e45756", "marker": "D"},
    "fuxi": {"label": "FuXi_WRF", "color": "#72b7b2", "marker": "v"},
    "aurora": {"label": "Aurora_WRF", "color": "#b279a2", "marker": "p"},
}

STORM_CONFIG = {
    "Ragasa": {
        "track_dir": INPUT_DIR / "track_interval1h_ragasa",
        "file_template": "track_d01_interval1h_{model}_{cycle}.txt",
        "best_track_file": INPUT_DIR / "best_track_Ragasa_JMA.xlsx",
        "time_step_hours": 1,
        "init_times": {
            "24h": pd.Timestamp("2025-09-22 22:00"),
            "48h": pd.Timestamp("2025-09-21 22:00"),
        },
        "end_time": pd.Timestamp("2025-09-24 10:00"),
        "closest_time": pd.Timestamp("2025-09-23 22:00"),
        "closest_label_offset": (95, 4),
        "expected_raw_points": {"24h": 37, "48h": 61},
        "expected_plot_points": {"24h": 6, "48h": 10},
    },
    "Yagi": {
        "track_dir": INPUT_DIR / "track_interval6h_yagi",
        "file_template": "track_yagi_{model}_{cycle}.txt",
        "best_track_file": INPUT_DIR / "best_track_Yagi_JMA.xlsx",
        "time_step_hours": 6,
        "init_times": {
            "24h": pd.Timestamp("2024-09-04 12:00"),
            "48h": pd.Timestamp("2024-09-03 12:00"),
        },
        "end_time": pd.Timestamp("2024-09-06 00:00"),
        "closest_time": pd.Timestamp("2024-09-05 12:00"),
        "closest_label_offset": (35, 48),
        "expected_raw_points": {"24h": 9, "48h": 13},
        "expected_plot_points": {"24h": 7, "48h": 11},
    },
}

STORM_ORDER = ("Ragasa", "Yagi")
CYCLES = ("24h", "48h")


def load_track_forecast(storm, model, cycle):
    config = STORM_CONFIG[storm]
    file_name = config["file_template"].format(model=model, cycle=cycle)
    values = np.atleast_2d(np.loadtxt(config["track_dir"] / file_name))
    lead_hour = np.arange(values.shape[0], dtype=int) * config["time_step_hours"]
    init_time = config["init_times"][cycle]

    return pd.DataFrame(
        {
            "storm": storm,
            "model": model,
            "cycle": cycle,
            "lead_hour": lead_hour,
            "init_time": init_time,
            "valid_time": init_time + pd.to_timedelta(lead_hour, unit="h"),
            "lon": values[:, 0],
            "lat": values[:, 1],
            "slp_hpa": values[:, 2],
        }
    )


track_forecasts = {
    storm: {
        cycle: {
            model: load_track_forecast(storm, model, cycle)
            for model in MODEL_STYLES
        }
        for cycle in CYCLES
    }
    for storm in STORM_ORDER
}


## Load JMA best tracks

In [ ]:
def load_best_track(file_path):
    return (
        pd.read_excel(file_path)
        .rename(
            columns={
                "Time": "time",
                "Lat": "lat_tenths",
                "Lon": "lon_tenths",
                "Pres": "pres_hpa",
                "Wind": "wind_kt",
            }
        )
        .assign(
            time=lambda df: pd.to_datetime(df["time"].astype(str), format="%y%m%d%H"),
            lat=lambda df: df["lat_tenths"] / 10,
            lon=lambda df: df["lon_tenths"] / 10,
        )
        [["time", "lat", "lon", "pres_hpa", "wind_kt"]]
        .sort_values("time")
        .reset_index(drop=True)
    )


best_tracks = {
    storm: load_best_track(config["best_track_file"])
    for storm, config in STORM_CONFIG.items()
}


## Prepare aligned tracks and plotting helpers

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.size"] = 16

HONG_KONG = {"lon": 114.1, "lat": 22.3}
MAP_CRS = ccrs.PlateCarree()
TEXT_BBOX = {"facecolor": "white", "alpha": 0.65, "edgecolor": "none", "pad": 1.5}
TEXT_STYLE = {"fontname": "Arial", "fontsize": 14}
TICK_STYLE = {"size": 16, "family": "Arial"}
LEGEND_STYLE = {"family": "Arial", "size": 12}


def prepare_storm_cycle_plot(storm, cycle):
    config = STORM_CONFIG[storm]
    forecast_start = config["init_times"][cycle]
    forecast_end = config["end_time"]
    forecast_tracks = track_forecasts[storm][cycle]

    best_track_window = (
        best_tracks[storm]
        .loc[lambda df: df["time"].between(forecast_start, forecast_end)]
        .copy()
        .sort_values("time")
    )
    aligned_times = pd.Index(best_track_window["time"])

    forecast_aligned = {
        model: (
            track.loc[
                track["valid_time"].between(forecast_start, forecast_end)
                & track["valid_time"].isin(aligned_times)
            ]
            .copy()
            .sort_values("valid_time")
        )
        for model, track in forecast_tracks.items()
    }

    return {
        "storm": storm,
        "cycle": cycle,
        "forecast_start": forecast_start,
        "forecast_end": forecast_end,
        "best_track": best_track_window,
        "forecast_aligned": forecast_aligned,
    }


def collect_plot_coordinates(storm_plots):
    lon_series = [pd.Series([HONG_KONG["lon"]])]
    lat_series = [pd.Series([HONG_KONG["lat"]])]

    for storm_plot in storm_plots:
        lon_series.append(storm_plot["best_track"]["lon"])
        lat_series.append(storm_plot["best_track"]["lat"])
        for forecast_track in storm_plot["forecast_aligned"].values():
            lon_series.append(forecast_track["lon"])
            lat_series.append(forecast_track["lat"])

    return (
        pd.concat(lon_series, ignore_index=True),
        pd.concat(lat_series, ignore_index=True),
    )


def compute_extent(
    storm_plots,
    fixed_pad=None,
    lon_pad_fraction=0.04,
    lat_pad_fraction=0.04,
    min_lon_pad=0.35,
    min_lat_pad=0.15,
):
    all_lon, all_lat = collect_plot_coordinates(storm_plots)
    if fixed_pad is None:
        lon_pad = max((all_lon.max() - all_lon.min()) * lon_pad_fraction, min_lon_pad)
        lat_pad = max((all_lat.max() - all_lat.min()) * lat_pad_fraction, min_lat_pad)
    else:
        lon_pad = lat_pad = fixed_pad

    return [
        all_lon.min() - lon_pad,
        all_lon.max() + lon_pad,
        all_lat.min() - lat_pad,
        all_lat.max() + lat_pad,
    ]


def match_extent_aspect(extent, target_aspect):
    west, east, south, north = extent
    lon_span = east - west
    lat_span = north - south
    current_aspect = lon_span / lat_span

    if current_aspect < target_aspect:
        extra = (lat_span * target_aspect - lon_span) / 2
        west -= extra
        east += extra
    elif current_aspect > target_aspect:
        extra = (lon_span / target_aspect - lat_span) / 2
        south -= extra
        north += extra

    return [west, east, south, north]


def add_map_base(ax, extent, show_longitude_labels=True):
    ax.set_extent(extent, crs=MAP_CRS)
    ax.add_feature(cfeature.LAND.with_scale("10m"), facecolor="#f7f3e8")
    ax.add_feature(cfeature.COASTLINE.with_scale("10m"), linewidth=0.6)

    gridliner = ax.gridlines(
        draw_labels=True, linewidth=0, color="none", alpha=0, linestyle="-"
    )
    gridliner.top_labels = False
    gridliner.right_labels = False
    gridliner.bottom_labels = show_longitude_labels
    gridliner.xlines = False
    gridliner.ylines = False
    gridliner.xlabel_style = TICK_STYLE
    gridliner.ylabel_style = TICK_STYLE
    return gridliner


def interpolate_track_position(best_track, target_time):
    before = best_track.loc[best_track["time"] <= target_time].iloc[-1]
    after = best_track.loc[best_track["time"] >= target_time].iloc[0]
    if before["time"] == after["time"]:
        return before["lon"], before["lat"]

    fraction = (target_time - before["time"]) / (after["time"] - before["time"])
    lon = before["lon"] + fraction * (after["lon"] - before["lon"])
    lat = before["lat"] + fraction * (after["lat"] - before["lat"])
    return lon, lat


def add_hong_kong_marker(ax, extent):
    west, east, south, north = extent
    lon_span = east - west
    lat_span = north - south

    if not (west <= HONG_KONG["lon"] <= east and south <= HONG_KONG["lat"] <= north):
        return

    ax.scatter(
        HONG_KONG["lon"],
        HONG_KONG["lat"],
        color="#d62728",
        marker="*",
        s=180,
        facecolor="#d62728",
        edgecolor="white",
        linewidth=0.8,
        transform=MAP_CRS,
        zorder=7,
    )
    ax.text(
        HONG_KONG["lon"] + 0.02 * lon_span,
        HONG_KONG["lat"] - 0.04 * lat_span,
        "Hong Kong",
        color="#d62728",
        transform=MAP_CRS,
        zorder=7,
        bbox=TEXT_BBOX,
        **TEXT_STYLE,
    )


def add_closest_time_annotation(ax, best_track, closest_time, label_offset):
    closest_lon, closest_lat = interpolate_track_position(best_track, closest_time)
    ax.scatter(
        closest_lon,
        closest_lat,
        s=90,
        facecolors="none",
        edgecolors="black",
        linewidth=1.2,
        transform=MAP_CRS,
        zorder=8,
    )
    ax.annotate(
        closest_time.strftime("%Y-%m-%d %HZ"),
        xy=(closest_lon, closest_lat),
        xytext=label_offset,
        textcoords="offset points",
        ha="left",
        va="bottom",
        color="black",
        transform=MAP_CRS,
        arrowprops={
            "arrowstyle": "<-",
            "linestyle": "--",
            "color": "black",
            "linewidth": 1.0,
            "shrinkA": 4,
            "shrinkB": 5,
        },
        clip_on=True,
        annotation_clip=True,
        zorder=6,
        bbox=TEXT_BBOX,
        **TEXT_STYLE,
    )


def plot_track_panel(ax, storm_plot, extent, panel_label, show_longitude_labels):
    add_map_base(ax, extent, show_longitude_labels=show_longitude_labels)
    config = STORM_CONFIG[storm_plot["storm"]]
    best_track_window = storm_plot["best_track"]

    ax.plot(
        best_track_window["lon"],
        best_track_window["lat"],
        color="black",
        linewidth=1.0,
        marker="o",
        markersize=4.0,
        label="Best track",
        transform=MAP_CRS,
        zorder=5,
    )

    for model, forecast_track in storm_plot["forecast_aligned"].items():
        style = MODEL_STYLES[model]
        ax.plot(
            forecast_track["lon"],
            forecast_track["lat"],
            color=style["color"],
            linewidth=1.0,
            marker=style["marker"],
            markersize=4.0,
            label=style["label"],
            transform=MAP_CRS,
            zorder=4,
        )

    add_hong_kong_marker(ax, extent)
    add_closest_time_annotation(
        ax,
        best_track_window,
        config["closest_time"],
        config["closest_label_offset"],
    )
    ax.text(
        0.02,
        0.96,
        panel_label,
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontname="Arial",
        fontsize=16,
        fontweight="bold",
        bbox=TEXT_BBOX,
        zorder=10,
    )


def plot_storm_figure(storm, storm_plots, extent, output_name):
    aspect = (extent[1] - extent[0]) / (extent[3] - extent[2])
    panel_height = 7.0 / aspect
    figure_height = max(2 * panel_height + 1.0, 4.5)
    fig, axes = plt.subplots(
        nrows=2,
        ncols=1,
        figsize=(8, figure_height),
        gridspec_kw={"height_ratios": [1, 1]},
        subplot_kw={"projection": MAP_CRS},
    )

    for index, cycle in enumerate(CYCLES):
        plot_track_panel(
            axes[index],
            storm_plots[cycle],
            extent,
            panel_label=f"{storm}_{cycle}",
            show_longitude_labels=index == len(CYCLES) - 1,
        )

    axes[0].legend(
        loc="upper right",
        ncol=1,
        columnspacing=1.0,
        handletextpad=0.5,
        labelspacing=0.4,
        frameon=True,
        prop=LEGEND_STYLE,
    )
    fig.subplots_adjust(hspace=0.04, left=0.10, right=0.98, top=0.98, bottom=0.10)
    fig.savefig(OUTPUT_DIR / output_name, dpi=600, format="tif", bbox_inches="tight")
    plt.show()


## Validate data and generate figures

In [ ]:
def validate_track_data(storm_cycle_plots):
    for storm, config in STORM_CONFIG.items():
        for cycle in CYCLES:
            expected_raw = config["expected_raw_points"][cycle]
            expected_plot = config["expected_plot_points"][cycle]
            plotted_tracks = storm_cycle_plots[storm][cycle]["forecast_aligned"]

            for model, raw_track in track_forecasts[storm][cycle].items():
                plotted_track = plotted_tracks[model]
                assert len(raw_track) == expected_raw, (storm, cycle, model, len(raw_track))
                assert len(plotted_track) == expected_plot, (
                    storm, cycle, model, len(plotted_track)
                )
                assert not plotted_track[["lon", "lat"]].isna().any().any()

            if storm == "Yagi":
                sample_track = plotted_tracks["era5"]
                assert sample_track["valid_time"].iloc[0] == config["init_times"][cycle]
                assert sample_track["valid_time"].iloc[-1] == config["end_time"]

storm_cycle_plots = {
    storm: {cycle: prepare_storm_cycle_plot(storm, cycle) for cycle in CYCLES}
    for storm in STORM_ORDER
}
validate_track_data(storm_cycle_plots)

# Ragasa retains the original 1.5-degree margin. Yagi uses adaptive padding
# and then matches Ragasa's panel aspect ratio.
ragasa_extent = compute_extent(storm_cycle_plots["Ragasa"].values(), fixed_pad=1.5)
ragasa_aspect = (ragasa_extent[1] - ragasa_extent[0]) / (
    ragasa_extent[3] - ragasa_extent[2]
)
yagi_extent = match_extent_aspect(
    compute_extent(storm_cycle_plots["Yagi"].values()),
    ragasa_aspect,
)

figure_specs = {
    "Ragasa": (ragasa_extent, "track_forecast_ragasa.tif"),
    "Yagi": (yagi_extent, "track_forecast_yagi.tif"),
}
for storm, (extent, output_name) in figure_specs.items():
    plot_storm_figure(storm, storm_cycle_plots[storm], extent, output_name)
